# Day 3 – Module 1: LangChain Framework

**Topics covered:**
- LangChain architecture
- Prompt templates and chains
- Agents and tool integration
- Connecting LangChain with vector databases

**What you will do in this notebook:**
Build prompt templates, compose chains with LCEL (LangChain Expression Language), define tools and agents, and connect a vector store retriever to answer questions from documents. Every section maps back to the architecture diagram. An OpenAI API key is required for LLM calls; notebooks include a fallback for review without a key.

## 1. LangChain architecture

LangChain orchestrates LLMs, prompts, tools, and data sources (such as vector stores) into composable pipelines. The diagram below shows the core components and how they connect.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        LANGCHAIN ARCHITECTURE                               │
│                                                                             │
│  User Input                                                                 │
│       │                                                                     │
│       ▼                                                                     │
│  ┌──────────────────┐                                                       │
│  │  Prompt Template  │  ChatPromptTemplate.from_messages(...)               │
│  └────────┬─────────┘                                                       │
│           │                                                                 │
│           ▼                                                                 │
│  ┌──────────────────┐     ┌──────────────┐                                  │
│  │       LLM        │────▶│    Tools      │  @tool, bind_tools              │
│  │  (ChatOpenAI)    │◀────│  (search,     │                                 │
│  └────────┬─────────┘     │   calculator) │                                 │
│           │               └──────────────┘                                  │
│           ▼                                                                 │
│  ┌──────────────────┐     ┌──────────────┐                                  │
│  │  Output Parser    │     │  Retriever    │  VectorStore.as_retriever()     │
│  │ (StrOutputParser) │     │  (vector DB)  │                                │
│  └────────┬─────────┘     └──────────────┘                                  │
│           │                                                                 │
│           ▼                                                                 │
│       Response                                                              │
└─────────────────────────────────────────────────────────────────────────────┘

Chain (LCEL):   prompt | llm | parser
Agent flow:     User → Agent → [LLM reasoning ↔ Tool calls] → Final answer
RAG flow:       query → Retriever → docs → Prompt(context+question) → LLM → answer
```

Each block maps to a LangChain class. The sections below implement them one by one.

## Component reference

| Component | Purpose | Main API | Example |
|---|---|---|---|
| `ChatPromptTemplate` | Structure system + user messages | `from_messages`, `invoke` | `prompt.invoke({"question": "..."})` |
| `ChatOpenAI` | LLM interface (OpenAI-compatible) | `invoke`, `bind_tools` | `llm.invoke(messages)` |
| `StrOutputParser` | Extract plain text from LLM response | `invoke` | `parser.invoke(ai_message)` |
| `@tool` decorator | Define callable tools | `name`, `description`, `func` | `tool.invoke({"query": "..."})` |
| `VectorStoreRetriever` | Retrieve documents from vector DB | `as_retriever`, `invoke` | `retriever.invoke("query")` |
| `RunnableSequence` (LCEL) | Compose components with `\|` | `invoke`, `stream`, `batch` | `chain = prompt \| llm \| parser` |

## 2. Setup

In [ ]:
import json, os, warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

API_KEY = os.environ.get("OPENAI_API_KEY", "")

with open("config.json") as f:
    CONFIG = json.load(f)

print(f"LLM model   : {CONFIG['llm_model']}")
print(f"Embedding   : {CONFIG['embedding_model']}")
print(f"API key set : {bool(API_KEY)}")

## 3. Prompt templates

Prompt templates standardize how system instructions and user inputs are combined before being sent to the LLM. This implements the **Prompt Template** block in the architecture diagram.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# System instruction + user question template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise enterprise support assistant. Answer in one sentence."),
    ("user", "{question}"),
])

# Preview the formatted messages (no LLM call yet)
formatted = prompt.invoke({"question": "How do I reset my VPN?"})
for msg in formatted.messages:
    print(f"[{msg.type:>6}] {msg.content}")

The template has two variables implicitly: the system message is fixed, the user message takes `{question}`. Calling `invoke` produces a list of messages ready for the LLM.

## 4. Calling the LLM

This implements the **LLM** block. We use `ChatOpenAI` which works with any OpenAI-compatible endpoint.

In [ ]:
from langchain_openai import ChatOpenAI

if not API_KEY:
    print("No OPENAI_API_KEY set — using simulated responses for review.")

llm = ChatOpenAI(
    model=CONFIG["llm_model"],
    temperature=CONFIG["temperature"],
    api_key=API_KEY or "sk-placeholder",
)
print(f"Model: {llm.model_name}")

In [ ]:
# Direct LLM call with formatted messages
if API_KEY:
    response = llm.invoke(formatted.messages)
    print(response.content)
else:
    print("[Simulated] Go to Settings > VPN > Reset Connection to reset your VPN client.")

## 5. Output parser

`StrOutputParser` extracts plain text from the LLM's `AIMessage` response. This implements the **Output Parser** block.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# Demo: parse a message object into a string
from langchain_core.messages import AIMessage
demo_msg = AIMessage(content="Reset your VPN via Settings > VPN > Reset Connection.")
print(type(demo_msg).__name__, "→", type(parser.invoke(demo_msg)).__name__)
print(parser.invoke(demo_msg))

## 6. Chains with LCEL

LangChain Expression Language (LCEL) composes components using the `|` operator. A chain is a `RunnableSequence`: each component's output feeds into the next.

```
prompt | llm | parser
  ↓        ↓      ↓
Messages → AIMessage → str
```

This implements the full **Prompt → LLM → Parser** chain from the architecture diagram.

In [ ]:
chain = prompt | llm | parser

if API_KEY:
    answer = chain.invoke({"question": "What is LangChain used for?"})
    print(answer)
else:
    print("[Simulated] LangChain is a framework for building applications powered by LLMs.")

### Swapping components

In [ ]:
# Same chain structure, different prompt
technical_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a senior Python developer. Respond with code only, no explanation."),
    ("user", "{question}"),
])

code_chain = technical_prompt | llm | parser

if API_KEY:
    result = code_chain.invoke({"question": "Write a function that computes factorial recursively."})
    print(result)
else:
    print("[Simulated]\ndef factorial(n):\n    return 1 if n <= 1 else n * factorial(n - 1)")

The chain structure stays identical — only the prompt template changed. This composability is core to LangChain.

### Multi-step chain: translate then summarize

In [ ]:
from langchain_core.runnables import RunnablePassthrough

translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the following text to English. Return only the translation."),
    ("user", "{text}"),
])

summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following text in one sentence."),
    ("user", "{text}"),
])

# Chain 1: translate
translate_chain = translate_prompt | llm | parser

# Chain 2: take translation output, feed as {text} to summarize
full_chain = (
    translate_chain
    | (lambda translated: {"text": translated})
    | summarize_prompt
    | llm
    | parser
)

if API_KEY:
    result = full_chain.invoke({
        "text": "LangChain est un framework pour construire des applications alimentées par des modèles de langage."
    })
    print(result)
else:
    print("[Simulated] LangChain is a framework for building LLM-powered applications.")

### Batch processing

In [ ]:
questions = [
    {"question": "What is a prompt template?"},
    {"question": "What is LCEL?"},
    {"question": "What is a retriever?"},
]

if API_KEY:
    results = chain.batch(questions)
    for q, a in zip(questions, results):
        print(f"Q: {q['question']}")
        print(f"A: {a}\n")
else:
    simulated = [
        "A prompt template structures inputs into a message format for the LLM.",
        "LCEL is LangChain Expression Language for composing runnables with the pipe operator.",
        "A retriever fetches relevant documents from a data store for context.",
    ]
    for q, a in zip(questions, simulated):
        print(f"Q: {q['question']}")
        print(f"A: {a}\n")

### Streaming

In [ ]:
if API_KEY:
    print("Streamed response: ", end="")
    for chunk in chain.stream({"question": "Explain vector databases in two sentences."}):
        print(chunk, end="", flush=True)
    print()
else:
    print("Streamed response: Vector databases store and index high-dimensional vectors for fast similarity search.")

## 7. Tools and tool integration

Tools extend the LLM beyond text generation. An agent decides *which* tool to call based on the user's question. This implements the **Tools** block in the architecture diagram.

### Defining tools

Use the `@tool` decorator to turn any Python function into a LangChain tool. The docstring becomes the tool description that the LLM sees.

In [ ]:
from langchain_core.tools import tool

@tool
def search_knowledge_base(query: str) -> str:
    """Search the internal knowledge base for information about IT policies and procedures."""
    # Simulated knowledge base
    kb = {
        "vpn": "VPN access requires GlobalProtect client. Download from IT portal. Contact helpdesk for credentials.",
        "password": "Passwords must be 12+ characters. Reset via Settings > Security > Change Password.",
        "software": "Request software through ServiceNow catalog. Approval takes 1-2 business days.",
        "mfa": "Multi-factor authentication uses Microsoft Authenticator. Enable in Azure AD settings.",
    }
    query_lower = query.lower()
    for key, value in kb.items():
        if key in query_lower:
            return value
    return "No matching article found. Please contact the helpdesk."

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result. Input must be a valid Python math expression."""
    try:
        # Only allow safe math operations
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return "Error: only numeric expressions with +, -, *, /, () are allowed."
        result = eval(expression, {"__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

print("Tools defined:")
for t in [search_knowledge_base, calculator]:
    print(f"  {t.name}: {t.description[:60]}...")

### Tool schemas

In [ ]:
# The LLM uses tool schemas to decide when and how to call each tool
import json as _json

for t in [search_knowledge_base, calculator]:
    schema = t.get_input_schema().model_json_schema()
    print(f"Tool: {t.name}")
    print(f"  Schema: {_json.dumps(schema, indent=2)}")
    print()

### Binding tools to the LLM

`bind_tools` tells the LLM which tools are available. The LLM returns structured `tool_calls` when it decides a tool is needed.

In [ ]:
tools = [search_knowledge_base, calculator]
llm_with_tools = llm.bind_tools(tools)

if API_KEY:
    # The LLM should decide to call the calculator
    response = llm_with_tools.invoke("What is 1024 * 768?")
    print(f"Content: {response.content}")
    print(f"Tool calls: {response.tool_calls}")
else:
    print("Content: ")
    print("Tool calls: [{'name': 'calculator', 'args': {'expression': '1024 * 768'}, 'id': 'call_abc123'}]")

In [ ]:
if API_KEY:
    # The LLM should decide to call search_knowledge_base
    response = llm_with_tools.invoke("How do I set up VPN access?")
    print(f"Content: {response.content}")
    print(f"Tool calls: {response.tool_calls}")
else:
    print("Content: ")
    print("Tool calls: [{'name': 'search_knowledge_base', 'args': {'query': 'VPN access'}, 'id': 'call_def456'}]")

### Building a ReAct agent

An agent uses the LLM to reason about which tool to call, executes the tool, observes the result, and repeats until it has a final answer.

```
User question → LLM thinks → calls Tool → observes result → LLM answers
```

This implements the **Agent flow** from the architecture diagram.

In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm, tools)

if API_KEY:
    result = agent.invoke({"messages": [("user", "What is 256 * 384? And how do I reset my password?")]})
    for msg in result["messages"]:
        role = getattr(msg, "type", "unknown")
        content = getattr(msg, "content", "")
        if content:
            print(f"[{role}] {content[:200]}")
else:
    print("[human] What is 256 * 384? And how do I reset my password?")
    print("[ai] Let me help with both questions.")
    print("[tool] 98304")
    print("[tool] Passwords must be 12+ characters. Reset via Settings > Security > Change Password.")
    print("[ai] 256 * 384 = 98,304. To reset your password, go to Settings > Security > Change Password.")

The agent made two tool calls — `calculator` and `search_knowledge_base` — then combined the results into a single answer. The LLM decided which tool to use based on each part of the question.

### Enterprise scenario: IT support agent with ticket creation

In [ ]:
import datetime

@tool
def create_ticket(summary: str, priority: str = "medium") -> str:
    """Create an IT support ticket. Priority must be low, medium, or high."""
    ticket_id = f"TKT-{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"
    return f"Ticket {ticket_id} created. Summary: {summary}. Priority: {priority}."

support_tools = [search_knowledge_base, calculator, create_ticket]
support_agent = create_react_agent(llm, support_tools)

if API_KEY:
    result = support_agent.invoke({
        "messages": [("user", "My VPN is not connecting and I've tried restarting. Please create a high-priority ticket.")]
    })
    for msg in result["messages"]:
        content = getattr(msg, "content", "")
        if content:
            print(f"[{msg.type}] {content[:300]}")
else:
    print("[human] My VPN is not connecting and I've tried restarting. Please create a high-priority ticket.")
    print("[tool] VPN access requires GlobalProtect client. Download from IT portal. Contact helpdesk for credentials.")
    print("[tool] Ticket TKT-20260310120000 created. Summary: VPN not connecting. Priority: high.")
    print("[ai] I found that VPN requires the GlobalProtect client. A high-priority ticket has been created.")

## 8. Connecting LangChain with vector databases

This implements the **Retriever** block. We build a small vector store, create a retriever, and use it in a chain that answers questions from retrieved documents. This is a preview of the RAG pattern covered in depth in Module 2.

### Building a vector store from documents

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Small document set — internal IT policies
documents = [
    "All employees must complete annual cybersecurity training by December 31. Failure to complete results in restricted network access.",
    "Remote work requires VPN connection via GlobalProtect. Request access through the IT ServiceNow portal.",
    "Software installations require approval from the IT department. Submit requests via the ServiceNow catalog.",
    "Password policy: minimum 12 characters, must include uppercase, lowercase, number, and special character. Rotate every 90 days.",
    "Multi-factor authentication (MFA) is mandatory for all cloud services. Use Microsoft Authenticator app.",
    "Data classification levels: Public, Internal, Confidential, Restricted. All customer data is Confidential or above.",
    "Incident response: report security incidents to security@company.com within 1 hour of discovery.",
    "Laptop encryption is mandatory. BitLocker (Windows) or FileVault (macOS) must be enabled.",
    "Guest WiFi is on a separate network. Do not share corporate WiFi credentials with visitors.",
    "USB storage devices are blocked on corporate laptops. Use approved cloud storage for file transfers.",
]

embeddings = HuggingFaceEmbeddings(model_name=CONFIG["embedding_model"])

vectorstore = Chroma.from_texts(
    texts=documents,
    embedding=embeddings,
    collection_name=CONFIG["chroma_collection_name"],
)
print(f"Vector store created with {vectorstore._collection.count()} documents")

### Creating a retriever

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": CONFIG["retriever_top_k"]})

# Test retrieval
query = "What are the password requirements?"
retrieved_docs = retriever.invoke(query)

print(f"Query: {query}")
print(f"Retrieved {len(retrieved_docs)} documents:\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"  {i}. {doc.page_content[:100]}...")

### RAG chain: retriever + prompt + LLM

This combines the retriever with a prompt and LLM into a single chain — the **RAG flow** from the architecture diagram:

```
query → Retriever → docs → format_docs → Prompt(context + question) → LLM → answer
```

In [ ]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the provided context. If the answer is not in the context, say 'Not found in context.'"),
    ("user", "Context:\n{context}\n\nQuestion: {question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | parser
)

test_questions = [
    "What are the password requirements?",
    "How do I report a security incident?",
    "Can I use a USB drive on my corporate laptop?",
]

for q in test_questions:
    if API_KEY:
        answer = rag_chain.invoke(q)
    else:
        answers_sim = {
            "What are the password requirements?": "Passwords must be minimum 12 characters with uppercase, lowercase, number, and special character, rotated every 90 days.",
            "How do I report a security incident?": "Report security incidents to security@company.com within 1 hour of discovery.",
            "Can I use a USB drive on my corporate laptop?": "No, USB storage devices are blocked on corporate laptops. Use approved cloud storage for file transfers.",
        }
        answer = answers_sim.get(q, "[Simulated answer]")
    print(f"Q: {q}")
    print(f"A: {answer}\n")

### Returning answers with source documents

In [ ]:
from langchain_core.runnables import RunnableLambda

def retrieve_with_sources(question):
    docs = retriever.invoke(question)
    return {
        "context": format_docs(docs),
        "question": question,
        "sources": [doc.page_content[:80] for doc in docs],
    }

rag_with_sources = (
    RunnableLambda(retrieve_with_sources)
    | RunnablePassthrough.assign(
        answer=lambda x: (rag_prompt | llm | parser).invoke(
            {"context": x["context"], "question": x["question"]}
        ) if API_KEY else "[Simulated] Answer based on retrieved context."
    )
)

if API_KEY:
    result = rag_with_sources.invoke("What training must employees complete?")
else:
    result = {
        "context": documents[0],
        "question": "What training must employees complete?",
        "sources": [documents[0][:80]],
        "answer": "[Simulated] All employees must complete annual cybersecurity training by December 31.",
    }

print(f"Question: {result['question']}")
print(f"Answer: {result['answer']}")
print(f"Sources:")
for s in result["sources"]:
    print(f"  - {s}...")

### Loading documents with LangChain document loaders

In [ ]:
from langchain_core.documents import Document

# Wrap raw texts as Document objects with metadata
doc_objects = [
    Document(page_content=text, metadata={"source": f"policy-{i+1}", "category": "IT"})
    for i, text in enumerate(documents)
]

print(f"Created {len(doc_objects)} Document objects")
print(f"Sample metadata: {doc_objects[0].metadata}")
print(f"Sample content:  {doc_objects[0].page_content[:80]}...")

### Vector store with metadata

In [ ]:
# Rebuild store with Document objects (includes metadata)
vectorstore_meta = Chroma.from_documents(
    documents=doc_objects,
    embedding=embeddings,
    collection_name="policies_with_meta",
)

retriever_meta = vectorstore_meta.as_retriever(search_kwargs={"k": 2})
results = retriever_meta.invoke("encryption requirements")

for doc in results:
    print(f"[{doc.metadata.get('source', 'unknown')}] {doc.page_content[:100]}...")

## 9. Agentic flow: combining tools and retrieval

An agent can use both tools and a retriever. The LLM decides whether to search the knowledge base, call a tool, or answer directly.

```
User → Agent → [LLM reasoning ↔ search_kb / calculator / create_ticket] → Final answer
```

In [ ]:
@tool
def search_policies(query: str) -> str:
    """Search company IT policies for relevant information. Use this for any policy, security, or compliance question."""
    docs = retriever.invoke(query)
    if docs:
        return "\n".join(f"- {d.page_content}" for d in docs[:3])
    return "No matching policies found."

enterprise_agent = create_react_agent(llm, [search_policies, calculator, create_ticket])

if API_KEY:
    result = enterprise_agent.invoke({
        "messages": [("user", "What is the data classification for customer data? Also, what is 365 * 24?")]
    })
    for msg in result["messages"]:
        content = getattr(msg, "content", "")
        if content:
            print(f"[{msg.type}] {content[:300]}")
else:
    print("[human] What is the data classification for customer data? Also, what is 365 * 24?")
    print("[tool] - Data classification levels: Public, Internal, Confidential, Restricted. All customer data is Confidential or above.")
    print("[tool] 8760")
    print("[ai] Customer data is classified as Confidential or Restricted. And 365 * 24 = 8,760 hours in a year.")

## 10. Chain patterns comparison

| Pattern | Components | When to use |
|---|---|---|
| Simple chain | `prompt \| llm \| parser` | Direct question → answer |
| Multi-step chain | `chain1 \| transform \| chain2` | Translate → summarize, extract → classify |
| RAG chain | `retriever \| format \| prompt \| llm \| parser` | Answer from documents |
| Agent | `create_react_agent(llm, tools)` | Dynamic tool selection, multi-step reasoning |
| Agentic RAG | Agent with retriever as tool | Mixed: tools + document Q&A |

## 11. Error handling in production chains

In [ ]:
from langchain_core.runnables import RunnableLambda

def safe_invoke(chain, inputs, fallback="Unable to process request."):
    try:
        return chain.invoke(inputs)
    except Exception as e:
        return f"{fallback} Error: {type(e).__name__}: {e}"

# Example: invoke with error handling
if API_KEY:
    result = safe_invoke(chain, {"question": "What is LangChain?"})
else:
    result = safe_invoke(
        RunnableLambda(lambda x: "[Simulated] LangChain orchestrates LLMs into applications."),
        {"question": "What is LangChain?"}
    )
print(result)

In [ ]:
# LangChain built-in fallback: with_fallbacks
fallback_chain = RunnableLambda(lambda x: "Service temporarily unavailable. Please try again.")

robust_chain = chain.with_fallbacks([fallback_chain])
print("Robust chain created with fallback")
print(f"Type: {type(robust_chain).__name__}")

## 12. Chain inspection and timing

In [ ]:
import time

# Inspect chain structure
print("Chain steps:")
for i, step in enumerate(chain.steps if hasattr(chain, 'steps') else [chain]):
    print(f"  {i+1}. {type(step).__name__}")

# Timing
if API_KEY:
    start = time.perf_counter()
    _ = chain.invoke({"question": "What is the capital of France?"})
    elapsed = time.perf_counter() - start
    print(f"\nChain execution time: {elapsed:.2f}s")
else:
    print("\nChain execution time: ~0.5-2.0s (depends on API latency)")

## Summary

- **Prompt templates** structure system and user messages with variables
- **Chains (LCEL)** compose components with `|` — prompt, LLM, parser, and any custom step
- **Tools and agents** let the LLM decide when to call external functions (search, calculator, ticket creation)
- **Retriever** connects a vector store to the chain for document-aware Q&A (RAG preview)
- **Agentic RAG** combines tools and retrieval in a single agent

Next: **Module 2 — RAG Fundamentals** goes deeper into retrieval-augmented generation patterns, chunking strategies, and hybrid retrieval.

## Cleanup

In [ ]:
# Remove temporary Chroma collections
try:
    vectorstore._client.delete_collection(CONFIG["chroma_collection_name"])
    vectorstore_meta._client.delete_collection("policies_with_meta")
    print("Temporary vector store collections removed.")
except Exception:
    pass

print("Notebook complete.")